In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('../data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

../data\cleaned_data.csv
../data\cleaned_test.csv
../data\preprocessed_data.csv
../data\preprocessed_test.csv
../data\probabilities_nn.csv
../data\test.csv
../data\train.csv


In [4]:
import pandas as pd
import numpy as np
# import ydf  

pd.set_option('future.no_silent_downcasting', True)

train_df = pd.read_csv('../data/preprocessed_data.csv')
test_df = pd.read_csv('../data/preprocessed_test.csv')

train_df["Side"] = train_df["Side"].map({"P": 0, "S": 1})
test_df["Side"] = test_df["Side"].map({"P": 0, "S": 1})

selected_columns_train = [
    "CryoSleep", "LuxurySpendings", "Age", "VIP", "RoomService", 
    "MainSpendings", "VRDeck", "Cabin_num", "Side",
    "CabinMidFlag", "HomePlanetEarth", "HomePlanetEuropa",
    "HomePlanetMars", "Destination55 Cancri e",
    "DestinationPSO J318.5-22", "DestinationTRAPPIST-1e",
    "Group1", "Group2", "Group3", "Group4", "Group5",
    "Group6", "Group7", "DeckA", "DeckB", "DeckC",
    "DeckD", "DeckE", "DeckF", "DeckG",
    "FamilyMembersCabinCount", "avg_spending_family","Transported"
]
selected_columns_test = [
    "CryoSleep", "LuxurySpendings", "Age", "VIP", "RoomService", 
    "MainSpendings", "VRDeck", "Cabin_num", "Side",
    "CabinMidFlag", "HomePlanetEarth", "HomePlanetEuropa",
    "HomePlanetMars", "Destination55 Cancri e",
    "DestinationPSO J318.5-22", "DestinationTRAPPIST-1e",
    "Group1", "Group2", "Group3", "Group4", "Group5",
    "Group6", "Group7", "DeckA", "DeckB", "DeckC",
    "DeckD", "DeckE", "DeckF", "DeckG",
    "FamilyMembersCabinCount", "avg_spending_family"
]


C:\Users\masar\AppData\Local\Temp\ipykernel_82696\1720611138.py:5: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


In [6]:
import lightgbm as lgbm
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.optimize import minimize
from sklearn.metrics import log_loss

X = train_df[selected_columns_test]
y = train_df['Transported'].astype(int)
X_t = test_df[selected_columns_test]

# Store aggregated predictions across multiple seeds
model_keys = ['lgb', 'xgb', 'cat', 'hgb', 'et', "rf"]#, "ydf"]
oof_preds = {m: np.zeros(len(X)) for m in model_keys}
test_preds = {m: np.zeros(len(X_t)) for m in model_keys}

SEEDS = [42, 123, 456, 789, 1024, 2048, 2024]
N_FOLDS = 10

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    print(f'[STATUS] Training Ensemble for Seed {seed}...')
    
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[trn_idx], y.iloc[trn_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        # 1. LightGBM
        m_lgb = lgbm.LGBMClassifier(n_estimators=700, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=seed+fold, verbosity=-1)
        m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgbm.early_stopping(100, verbose=False)])
        oof_preds['lgb'][val_idx] += m_lgb.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_preds['lgb'] += m_lgb.predict_proba(X_t)[:, 1] / (N_FOLDS * len(SEEDS))
        
        # # 2. XGBoost (Tuned from 0.814 Optuna Reference)
        # m_xgb = xgb.XGBClassifier(n_estimators=700, learning_rate=0.05, max_depth=5, subsample=0.96, colsample_bytree=0.93, reg_lambda=3.06, reg_alpha=4.582, random_state=seed+fold, eval_metric='logloss', early_stopping_rounds=100, verbosity=0, tree_method='hist')
        # m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        # oof_preds['xgb'][val_idx] += m_xgb.predict_proba(X_va)[:, 1] / len(SEEDS)
        # test_preds['xgb'] += m_xgb.predict_proba(X_t)[:, 1] / (N_FOLDS * len(SEEDS))
        
        # 3. CatBoost
        m_cat = CatBoostClassifier(iterations=700, learning_rate=0.03, depth=6, random_seed=seed+fold, verbose=0, od_type='Iter', od_wait=100)
        m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va))
        oof_preds['cat'][val_idx] += m_cat.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_preds['cat'] += m_cat.predict_proba(X_t)[:, 1] / (N_FOLDS * len(SEEDS))
        
        # 4. HistGradientBoosting
        m_hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.04, max_depth=6, random_state=seed+fold)
        m_hgb.fit(X_tr, y_tr)
        oof_preds['hgb'][val_idx] += m_hgb.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_preds['hgb'] += m_hgb.predict_proba(X_t)[:, 1] / (N_FOLDS * len(SEEDS))
        
        # 5. ExtraTrees
        m_et = ExtraTreesClassifier(n_estimators=500, max_depth=8, min_samples_split=4, random_state=seed+fold, n_jobs=-1)
        m_et.fit(X_tr.fillna(-1), y_tr)
        oof_preds['et'][val_idx] += m_et.predict_proba(X_va.fillna(-1))[:, 1] / len(SEEDS)
        test_preds['et'] += m_et.predict_proba(X_t.fillna(-1))[:, 1] / (N_FOLDS * len(SEEDS))
        
        # 6. Random Forest
        m_rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            random_state=seed+fold,
            n_jobs=-1
        )
        m_rf.fit(X_tr.fillna(-1), y_tr)
        oof_preds['rf'][val_idx] += m_rf.predict_proba(X_va.fillna(-1))[:, 1] / len(SEEDS)
        test_preds['rf'] += m_rf.predict_proba(X_t.fillna(-1))[:, 1] / (N_FOLDS * len(SEEDS))
        
        X_tr_ydf = X_tr.copy()
        X_tr_ydf['target_label'] = y_tr

        # 7. GradientBoostedTreesLearner
        # m_ydf = ydf.GradientBoostedTreesLearner(
        #     label="target_label",
        #     num_trees=500,
        #     growing_strategy="BEST_FIRST_GLOBAL", 
        #     max_depth=6,
        #     shrinkage=0.05,
        #     random_seed=seed + fold
        # )
        
        # model_ydf = m_ydf.train(X_tr_ydf)
        
        # # YDF returns predictions as a 1D array of probabilities for the positive class 
        # # in binary classification (or a 2D array if multiclass).
        # val_p = model_ydf.predict(X_va)
        # test_p = model_ydf.predict(X_t)
        
        # oof_preds['ydf'][val_idx] += val_p / len(SEEDS)
        # test_preds['ydf'] += test_p / (N_FOLDS * len(SEEDS))
        
print('[SUCCESS] Multi-Seed OOF aggregation complete.')


[STATUS] Training Ensemble for Seed 42...
[STATUS] Training Ensemble for Seed 123...
[STATUS] Training Ensemble for Seed 456...
[STATUS] Training Ensemble for Seed 789...
[STATUS] Training Ensemble for Seed 1024...
[STATUS] Training Ensemble for Seed 2048...
[STATUS] Training Ensemble for Seed 2024...
[SUCCESS] Multi-Seed OOF aggregation complete.


In [7]:
# 1. Minimized the log-loss of the weighted average
def objective(w, oofs, y_real):
    blend = np.average(oofs, axis=1, weights=w)
    return log_loss(y_real, blend)
    
X_oofs = np.column_stack([oof_preds[m] for m in model_keys])
X_tests = np.column_stack([test_preds[m] for m in model_keys])

constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
bounds = [(0, 1)] * len(model_keys)
init_w = [1.0/len(model_keys)] * len(model_keys)

opt_res = minimize(objective, init_w, args=(X_oofs, y), bounds=bounds, constraints=constraints, method='SLSQP')
best_w = opt_res.x
print(f'[SUCCESS] Optimal Blend Weights: {dict(zip(model_keys, np.round(best_w, 4)))}')

stack_oof = np.average(X_oofs, axis=1, weights=best_w)
stack_test = np.average(X_tests, axis=1, weights=best_w)

# 2. Find the best threshold
best_threshold = 0.5
best_acc = 0
for threshold in np.linspace(0.30, 0.70, 401):
    acc = accuracy_score(y, stack_oof >= threshold)
    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold
        
final_test_labels = stack_test >= best_threshold

[SUCCESS] Optimal Blend Weights: {'lgb': np.float64(0.1459), 'xgb': np.float64(0.0), 'cat': np.float64(0.6821), 'hgb': np.float64(0.1721), 'et': np.float64(0.0), 'rf': np.float64(0.0)}


In [8]:
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,DeckG,DeckT,DeckG_midCabin,DeckE_midCabin,DeckF_midCabin,NoSpending_notCryo,total_spending_family,avg_spending_family,LuxurySpendings,MainSpendings
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0,0,0,...,0,0,0,0,0,1,0,0.0,0,0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109,9,25,...,0,0,0,0,0,0,736,736.0,593,134
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43,3576,0,...,0,0,0,0,0,0,15559,7779.5,6764,43
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0,1283,371,...,0,0,0,0,0,0,15559,7779.5,3522,371
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303,70,151,...,0,0,0,0,0,0,1091,1091.0,567,454


In [9]:
output = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': final_test_labels
})

output.to_csv('../data/submission2.csv', index=False)
print("Finished")

Finished


In [10]:
best_w

array([1.45880993e-01, 0.00000000e+00, 6.82067122e-01, 1.72051885e-01,
       3.06996599e-17, 0.00000000e+00])

In [11]:
best_threshold

np.float64(0.524)

In [12]:
best_w

array([1.45880993e-01, 0.00000000e+00, 6.82067122e-01, 1.72051885e-01,
       3.06996599e-17, 0.00000000e+00])